# Leaf Disease Detection using TensorFlow and Transfer Learning

This notebook demonstrates how to build a convolutional neural network (CNN) to detect diseases in plant leaves using the **PlantVillage** dataset. We will use Transfer Learning with the **MobileNetV2** architecture for efficient training and inference.

**Features:**
- Automated dataset download (no API keys needed)
- Data Augmentation
- Transfer Learning (MobileNetV2)
- Comprehensive Evaluation (Confusion Matrix, Classification Report)
- Inference on custom images

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
import matplotlib.pyplot as plt
import numpy as np
import os
import pathlib
import random
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow version:", tf.__version__)

## 1. Dataset Preparation

We will clone the PlantVillage dataset directly from GitHub. This allows the notebook to be fully reproducible without manual file uploads.

In [ ]:
# Clone the dataset if not already present
if not os.path.exists("PlantVillage-Dataset"):
    !git clone https://github.com/spMohanty/PlantVillage-Dataset
else:
    print("Dataset already cloned.")

# Set the dataset directory
dataset_dir = pathlib.Path("PlantVillage-Dataset/raw/color")

# Verify the download
image_count = len(list(dataset_dir.glob('*/*.JPG'))) + len(list(dataset_dir.glob('*/*.jpg')))
print(f"Total images downloaded: {image_count}")

In [ ]:
BATCH_SIZE = 32
IMG_SIZE = (160, 160)

print("Loading training set...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("Loading validation set...")
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
print(f"Classes found: {len(class_names)}")
print(class_names[:5])

In [ ]:
# Visualize some examples
plt.figure(figsize=(10, 10))
for images, labels in train_dataset.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

## 2. Model Building (Transfer Learning)

We use MobileNetV2 pretrained on ImageNet. We freeze the base layers and add a classification head.

In [ ]:
# Data Augmentation Layer
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal'),
  tf.keras.layers.RandomRotation(0.2),
])

# MobileNetV2 preprocessing
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

IMG_SHAPE = IMG_SIZE + (3,)
base_model = MobileNetV2(input_shape=IMG_SHAPE,
                         include_top=False,
                         weights='imagenet')

base_model.trainable = False
base_model.summary()

In [ ]:
# Build the full model
inputs = tf.keras.Input(shape=IMG_SHAPE)
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])
model.summary()

## 3. Training

In [ ]:
epochs = 10
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=epochs
)

## 4. Evaluation

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Loss')
plt.show()

### Confusion Matrix & Classification Report

In [ ]:
# Get predictions for validation set
y_true = []
y_pred = []

for images, labels in validation_dataset:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(20, 20))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## 5. Inference (Predict on New Image)

In [ ]:
def predict_image(image_path):
    img = tf.keras.utils.load_img(
        image_path, target_size=IMG_SIZE
    )
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0) # Create a batch

    predictions = model.predict(img_array)
    score = predictions[0] # Probability distribution

    print(
        "This image most likely belongs to {} with a {:.2f} percent confidence."
        .format(class_names[np.argmax(score)], 100 * np.max(score))
    )
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# Run prediction on a random sample image from the dataset
print("Testing prediction on a random sample image...")
all_image_paths = list(dataset_dir.glob('*/*'))
if all_image_paths:
    random_image_path = random.choice(all_image_paths)
    print(f'Predicting on: {random_image_path}')
    predict_image(random_image_path)

# Example usage with a file upload (Commented out)
# from google.colab import files
# uploaded = files.upload()
# for fn in uploaded.keys():
#   predict_image(fn)

In [ ]:
# Save the entire model
model.save('leaf_disease_model_final.h5')